In [1]:
import os
import sys

In [2]:
python_path = sys.executable

In [3]:
os.environ['PYSPARK_PYTHON'] = python_path
os.environ['PYSPARK_DRIVER_PYTHON'] = python_path

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [5]:
spark = (
    SparkSession.builder.appName('Pyspark_intro')
    .master('local[2]')
    .config('spark.pyspark.python', python_path)
    .config('spark.pyspark.driver.python', python_path)
    .config('spark.python.use.daemon', 'false')
    .config('spark.python.worker.faulthandler.enabled', 'true')
    .getOrCreate()
)

In [6]:
sc = spark.sparkContext

print(f"Tryb działania : {sc.master}")
print(f"Liczba rdzeni: {sc.defaultParallelism}")
print(f"Nazwa aplikacji: {sc.appName}")
print(f"Adres interfejsu Spark UI: {sc.uiWebUrl}")

Tryb działania : local[2]
Liczba rdzeni: 2
Nazwa aplikacji: Pyspark_intro
Adres interfejsu Spark UI: http://host.docker.internal:4040


In [7]:
d = [('Jan', 'Ksiegowosc', 25000),
     ('Tomasz', 'Ksiegowosc', 9000),
     ('Karolina', 'HR', 8000)]
c = ['Imie', 'Dzial', 'Pensja']

df = spark.createDataFrame(d, c)

In [8]:
print("Liczba partycji:", df.rdd.getNumPartitions())

Liczba partycji: 2


In [9]:
d_filtered = df.filter(F.col('Pensja') > 8500).groupBy('Dzial').agg(F.avg('Pensja'))

In [10]:
d_filtered.explain(True)

== Parsed Logical Plan ==
'Aggregate ['Dzial], ['Dzial, unresolvedalias('avg('Pensja))]
+- Filter (Pensja#2L > cast(8500 as bigint))
   +- LogicalRDD [Imie#0, Dzial#1, Pensja#2L], false

== Analyzed Logical Plan ==
Dzial: string, avg(Pensja): double
Aggregate [Dzial#1], [Dzial#1, avg(Pensja#2L) AS avg(Pensja)#7]
+- Filter (Pensja#2L > cast(8500 as bigint))
   +- LogicalRDD [Imie#0, Dzial#1, Pensja#2L], false

== Optimized Logical Plan ==
Aggregate [Dzial#1], [Dzial#1, avg(Pensja#2L) AS avg(Pensja)#7]
+- Project [Dzial#1, Pensja#2L]
   +- Filter (isnotnull(Pensja#2L) AND (Pensja#2L > 8500))
      +- LogicalRDD [Imie#0, Dzial#1, Pensja#2L], false

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Dzial#1], functions=[avg(Pensja#2L)], output=[Dzial#1, avg(Pensja)#7])
   +- Exchange hashpartitioning(Dzial#1, 200), ENSURE_REQUIREMENTS, [plan_id=23]
      +- HashAggregate(keys=[Dzial#1], functions=[partial_avg(Pensja#2L)], output=[Dzial#1, sum#10, count#11L])
   

In [11]:
d_filtered.show()

+----------+-----------+
|     Dzial|avg(Pensja)|
+----------+-----------+
|Ksiegowosc|    17000.0|
+----------+-----------+

